In [1]:
using Clapeyron, Metaheuristics, Printf

In [2]:
like_parameter = """Clapeyron Database File
PCSAFT Like Parameters [csvtype = like,grouptype = PCSAFT]
species,Mw,segment,sigma,epsilon,n_H,n_e
isobutyricacid,88.11,3.82489,3.243649195,265.2913742,1,1
"""
assoc_parameter = """Clapeyron Database File
PCSAFT Assoc Parameters [csvtype = assoc,grouptype = PCSAFT]
species1,site1,species2,site2,epsilon_assoc,bondvol
isobutyricacid,H,isobutyricacid,e,2000.0,0.03
"""


model = PCSAFT(["isobutyricacid"], userlocations = [like_parameter, assoc_parameter])
print(model.params.epsilon_assoc.values)

Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[2000.0]

In [3]:
# Run this ONCE to fix your CSV files
function fix_line_endings(filename)
    content = read(filename, String)
    fixed = replace(content, "\r\n" => "\n")
    write(filename, fixed)
    println("Fixed: $filename")
end

fix_line_endings("satp_isobutyricacid.csv")
fix_line_endings("rhol_isobutyricacid.csv")

Fixed: satp_isobutyricacid.csv
Fixed: rhol_isobutyricacid.csv


In [4]:
### Saturation pressure — output in Pa (SI)
function my_saturation_p(model::EoSModel, T::Float64)
    sat = saturation_pressure(model, T)
    return sat[1]   # Pa
end

# Liquid density — output in kg/m³
function my_liquid_density(model::EoSModel, T::Float64)
    sat  = saturation_pressure(model, T)
    Mw   = model.params.Mw[1]      # g/mol
    rhol = 1.0 / sat[2]            # mol/m³
    return rhol * Mw / 1000.0      # mol/m³ × g/mol ÷ 1000 = kg/m³
end

my_liquid_density (generic function with 1 method)

In [5]:
toestimate = [
    Dict(
        :param => :segment,
        :lower => 1.0,
        :upper => 10.0,
        :guess => 3.
    ),
    Dict(
        :param => :epsilon,
        :lower => 100.,
        :upper => 500.,
        :guess => 250.
    ),
    Dict(
        :param => :sigma,
        :factor => 1e-10,
        :lower => 2.0,
        :upper => 7.0,
        :guess => 5.
    ),
    Dict(
        :param   => :epsilon_assoc,
        :indices => (1,1),
        :lower   => 0.0,
        :upper   => 5000.0,
        :guess   => 2400.0
    ),
    Dict(
        :param   => :bondvol,
        :indices => (1,1),
        :lower   => 0.0,
        :upper   => 1.0,
        :guess   => 0.4
    )
]

5-element Vector{Dict{Symbol, Any}}:
 Dict(:upper => 10.0, :param => :segment, :guess => 3.0, :lower => 1.0)
 Dict(:upper => 500.0, :param => :epsilon, :guess => 250.0, :lower => 100.0)
 Dict(:factor => 1.0e-10, :param => :sigma, :upper => 7.0, :guess => 5.0, :lower => 2.0)
 Dict(:upper => 5000.0, :param => :epsilon_assoc, :indices => (1, 1), :guess => 2400.0, :lower => 0.0)
 Dict(:upper => 1.0, :param => :bondvol, :indices => (1, 1), :guess => 0.4, :lower => 0.0)

In [6]:
estimator, objective, x0, upper, lower = Estimation(
    model,
    toestimate,
    [
        "rhol_isobutyricacid.csv",
        "satp_isobutyricacid.csv",
    ]
)
 
println("Initial objective value: ", objective(x0))

Initial objective value: 1.2894621681773557


In [7]:
method = ECA(; options = Options(iterations = 10000000, seed = 999))
 
params_opt, model_opt = optimize(objective, estimator, method)

([3.8124881560718014, 266.2130995354196, 3.250418223533321, 2621.630001743787, 0.0015883269329909665], PCSAFT{BasicIdeal, Float64}("isobutyricacid"))

In [8]:
using CSV, DataFrames, Printf

function calculate_AAD(model, csv_file, property_func)
    df = CSV.read(csv_file, DataFrame, comment="#", skipto=4)
    
    input_col  = names(df)[1]          # first column = input (T)
    output_col = names(df)[2]          # second column = out_xxx (experimental)
    
    inputs   = df[!, input_col]
    exp_vals = df[!, output_col]
    
    println("\n=== AAD: $csv_file ===")
    @printf("%-10s  %-12s  %-12s  %-8s\n", input_col, "exp", "calc", "ARD%")
    
    errors = Float64[]
    for (i, x) in enumerate(inputs)
        calc = property_func(model, x)
        err  = abs(calc - exp_vals[i]) / abs(exp_vals[i]) * 100
        push!(errors, err)
        @printf("%-10.4f  %-12.6f  %-12.6f  %-8.4f\n", x, exp_vals[i], calc, err)
    end
    
    aard = sum(errors) / length(errors)
    @printf("AARD = %.4f%%\n", aard)
    return aard
end

calculate_AAD (generic function with 1 method)

In [9]:
aard_p   = calculate_AAD(model_opt, "satp_isobutyricacid.csv",   my_saturation_p)


=== AAD: satp_isobutyricacid.csv ===


┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593


Clapeyron Estimator  exp           calc          ARD%    
375.2100    14550.000000  14527.159242  0.1570  
382.1400    19580.000000  19743.028375  0.8326  
391.7700    29650.000000  29442.287442  0.7005  
399.4300    39720.000000  39643.807890  0.1918  
405.4400    49790.000000  49486.378532  0.6098  
410.7500    59850.000000  59718.851608  0.2191  
415.5100    69920.000000  70253.569024  0.4771  
419.7700    79990.000000  80876.112027  1.1078  
423.3600    90070.000000  90772.647954  0.7801  
424.8600    95100.000000  95177.808648  0.0818  
426.1600    100590.000000  99129.108628  1.4523  
AARD = 0.6009%


0.600910597117078

In [10]:
aard_p   = calculate_AAD(model_opt, "rhol_isobutyricacid.csv",   my_liquid_density)


=== AAD: rhol_isobutyricacid.csv ===
Clapeyron Estimator  exp           calc          ARD%    


┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593


273.1500    968.000000    967.090541    0.0940  
293.1400    948.500000    948.283262    0.0229  
313.1300    928.500000    929.238513    0.0795  
333.1200    908.600000    909.757172    0.1274  
353.1200    888.600000    889.702384    0.1241  
373.1200    868.000000    869.014804    0.1169  
393.1300    847.200000    847.649416    0.0530  
413.1300    826.600000    825.598524    0.1212  
433.1400    804.400000    802.784805    0.2008  
453.1400    780.800000    779.122217    0.2149  
473.1500    756.000000    754.403137    0.2112  
493.1600    728.300000    728.361263    0.0084  
513.1700    698.200000    700.599632    0.3437  
AARD = 0.1321%


0.13214416693327188